## Add imports, set api and create folders

In [ ]:
from datetime import datetime
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

import pandas as pd
import requests

#Setting up API
BASE_URL = "https://api.ratings.food.gov.uk"

HEADERS = {"x-api-version": "2",
           "Accept": "application/json",}

#Finding and creating folders
RAW_FOLDER = Path("../data/business/raw")
INTERIM_FOLDER = Path("../data/business/interim")

RAW_FOLDER.mkdir(parents=True, exist_ok=True)
INTERIM_FOLDER.mkdir(parents=True, exist_ok=True)

SNAPSHOT_DATE = "2026-07-23"

snapshot_path = RAW_FOLDER / f"london_fhrs_raw_{SNAPSHOT_DATE}.csv"

if snapshot_path.exists():
    raise FileExistsError("The saved snapshot already exists. Do not overwrite the study data.")

NOTE: This notebook downloads live FHRS records. The submitted analysis uses the snapshot dated 23 July 2026. Rerunning the API retrieval will not recreate the historical snapshot. 

Reuse the saved project data to reproduce the submitted analysis; new downloads should retain the actual acquisition date as the SNAPSHOT_DATE throughout the pipeline to create separate output files.

## Creating function for repeatable API request

In [3]:
def get_fhrs_data(endpoint: str, params: dict | None = None) -> dict:

    response = requests.get(
            f"{BASE_URL}/{endpoint.lstrip('/')}",
            headers = HEADERS,
            params=params,
            timeout=60,
        )

    response.raise_for_status()

    return response.json()

## Inspecting the response structure

Using a random authority to get response keys, metadata, column data and business types and id's to get information which could be used in the code to follow.

In [4]:
authority_payload = get_fhrs_data("Authorities")

authorities = pd.DataFrame(authority_payload["authorities"])

sample_authority = authorities.sample(n=1,random_state=42).iloc[0]

print("Sample authority:", sample_authority["Name"])

sample_payload = get_fhrs_data(
    "Establishments",
    params={
        "localAuthorityId": int(sample_authority["LocalAuthorityId"]),
        "pageNumber": 1,
        "pageSize": 5,
    },)

print("\nResponse keys:")
print(sample_payload.keys())

print("\nMetadata fields:")
print(sample_payload.get("meta", {}).keys())

establishment_sample = pd.json_normalize(sample_payload["establishments"], sep=".",)

print("\nEstablishment columns:")
print(establishment_sample.columns.tolist())
print(f"There are {len(establishment_sample.columns.tolist())} columns for each establishment record.")

establishment_sample.head()

business_type_payload = get_fhrs_data("BusinessTypes")

business_types = pd.DataFrame(business_type_payload["businessTypes"])

business_types

Sample authority: Isle of Wight

Response keys:
dict_keys(['establishments', 'meta', 'links'])

Metadata fields:
dict_keys(['dataSource', 'extractDate', 'itemCount', 'returncode', 'totalCount', 'totalPages', 'pageSize', 'pageNumber'])

Establishment columns:
['AddressLine1', 'AddressLine2', 'AddressLine3', 'AddressLine4', 'BusinessName', 'BusinessType', 'BusinessTypeID', 'ChangesByServerID', 'Distance', 'FHRSID', 'LocalAuthorityBusinessID', 'LocalAuthorityCode', 'LocalAuthorityEmailAddress', 'LocalAuthorityName', 'LocalAuthorityWebSite', 'NewRatingPending', 'Phone', 'PostCode', 'RatingDate', 'RatingKey', 'RatingValue', 'RightToReply', 'SchemeType', 'geocode.longitude', 'geocode.latitude', 'scores.Hygiene', 'scores.Structural', 'scores.ConfidenceInManagement']
There are 28 columns for each establishment record.


,BusinessTypeId,BusinessTypeName,links
0,-1,All,"[{'rel': 'self', 'href': 'https://api.ratings...."
1,7,Distributors/Transporters,"[{'rel': 'self', 'href': 'https://api.ratings...."
2,7838,Farmers/growers,"[{'rel': 'self', 'href': 'https://api.ratings...."
3,5,Hospitals/Childcare/Caring Premises,"[{'rel': 'self', 'href': 'https://api.ratings...."
4,7842,Hotel/bed & breakfast/guest house,"[{'rel': 'self', 'href': 'https://api.ratings...."
5,14,Importers/Exporters,"[{'rel': 'self', 'href': 'https://api.ratings...."
6,7839,Manufacturers/packers,"[{'rel': 'self', 'href': 'https://api.ratings...."
7,7846,Mobile caterer,"[{'rel': 'self', 'href': 'https://api.ratings...."
8,7841,Other catering premises,"[{'rel': 'self', 'href': 'https://api.ratings...."
9,7843,Pub/bar/nightclub,"[{'rel': 'self', 'href': 'https://api.ratings...."


## Identify and create filter for london authorities

In [5]:
london_authorities = authorities[
    authorities["RegionName"]
    .astype(str)
    .str.strip()
    .str.casefold()
    .eq("london")
].copy()

print(f"London authorities found: {len(london_authorities)}")

london_authorities[
    ["LocalAuthorityId", "Name", "EstablishmentCount"]
].sort_values("Name")

London authorities found: 33


,LocalAuthorityId,Name,EstablishmentCount
14,88,Barking and Dagenham,1448
15,89,Barnet,2867
23,90,Bexley,1772
37,91,Brent,2490
43,92,Bromley,2459
53,93,Camden,4275
71,95,City of London Corporation,1800
80,94,Croydon,3144
96,96,Ealing,3708
113,97,Enfield,2379


## Downloading setup

FHRS establishment records are provided in as pages of responses for each local authority, the setup is to create function for a serial download of the pages within each authority.

Both serial and multithreaded approaches will be considered and evaluated for the full dataset download to try to process local authorities concurrently.

In [6]:
def download_one_authority_data(authority_id: int, authority_name: str, page_size: int = 200) -> pd.DataFrame: 
    
    pages = []
    page_number = 1

    while True:
        payload = get_fhrs_data("Establishments", params={
            "localAuthorityId": authority_id,
            "pageNumber": page_number,
            "pageSize": page_size,
        },)

        records = payload.get("establishments", [])

        if not records:
            break

        pages.append(pd.json_normalize(records, sep="."))

        total_pages = int(payload.get("meta", {}).get("totalPages", 1))

        if page_number >= total_pages:
            break

        page_number = page_number + 1
        time.sleep(0.2)
    
    if not pages:
        print(f"{authority_name}: no records downloaded.")
        return pd.DataFrame()
    
    authority_data = pd.concat(pages, ignore_index= True)

    print(f"{authority_name}: {page_number}/{total_pages} pages successfully downloaded with {len(authority_data):,} records found.")
    
    return authority_data

## Test function on one authority

In [7]:
test_authority = london_authorities.sample(n=1,random_state=42).iloc[0]

test_data = download_one_authority_data(
    authority_id=int(test_authority["LocalAuthorityId"]),
    authority_name=str(test_authority["Name"])
)

print(f"Test authority: {test_authority['Name']}")

print(f"Records downloaded: {len(test_data)}")

test_data.head()

Wandsworth: 15/15 pages successfully downloaded with 2,897 records found.
Test authority: Wandsworth
Records downloaded: 2897


,AddressLine1,AddressLine2,AddressLine3,AddressLine4,BusinessName,BusinessType,BusinessTypeID,ChangesByServerID,Distance,FHRSID,...,RatingDate,RatingKey,RatingValue,RightToReply,SchemeType,geocode.longitude,geocode.latitude,scores.Hygiene,scores.Structural,scores.ConfidenceInManagement
0,Unit 035 Turbine Hall Battersea Power Station ...,,London,Wandsworth,% Arabica,Restaurant/Cafe/Canteen,1,0,None,1782317,...,2025-01-22T00:00:00,fhrs_5_en-gb,5,,FHRS,-0.145443,51.4815083,5.0,0.0,5.0
1,50 Upper Tooting Road,,London,Wandsworth,12th Street Burgers,Restaurant/Cafe/Canteen,1,0,None,1672713,...,2025-12-30T00:00:00,fhrs_5_en-gb,5,,FHRS,-0.1621142,51.4345722,NaN,NaN,NaN
2,87 - 89 St Johns Road,,London,Wandsworth,2 Love Tea & Coffee,Restaurant/Cafe/Canteen,1,0,None,1572586,...,2024-03-18T00:00:00,fhrs_5_en-gb,5,,FHRS,-0.1669638,51.4610569,5.0,5.0,5.0
3,291 - 293 Lavender Hill,,London,Wandsworth,2 Love Tea &Coffee,Restaurant/Cafe/Canteen,1,0,None,1573319,...,2025-01-14T00:00:00,fhrs_5_en-gb,5,,FHRS,-0.1657349,51.4637924,0.0,5.0,5.0
4,155 Northcote Road,,London,Wandsworth,21 Grams,Restaurant/Cafe/Canteen,1,0,None,1574044,...,2025-04-30T00:00:00,fhrs_5_en-gb,5,,FHRS,-0.16451,51.4553748,5.0,5.0,5.0


# Full London dataset download

A serial download was initially tested and took around 2 minutes 30 seconds.  
A multithreaded approach using three workers reduced download time to around 50 seconds.

The method using 3 workers was retained to improve efficiency while limiting the number of simultaneous requests to the public API to reduce the risk of 403 and 429 errors.

In [11]:
london_dataframes = []

with ThreadPoolExecutor(max_workers=3) as executor:

    future_downloads = {executor.submit(
            download_one_authority_data,
            int(authority.LocalAuthorityId),
            str(authority.Name)): str(authority.Name)

        for authority in london_authorities.itertuples(index=False)
    }

    for future in as_completed(future_downloads):
        authority_name = future_downloads[future]

        try:
            authority_data = future.result()

            if not authority_data.empty:
                london_dataframes.append(authority_data)

        except Exception as error:
            print(f"{authority_name} failed: {error}")

print(f"\nAuthorities successfully downloaded: {len(london_dataframes)}/{len(london_authorities)}")

if len(london_dataframes) != len(london_authorities):
    raise RuntimeError("Some authorities failed to download. Do not save the snapshot yet.")

london_fhrs_raw = pd.concat(london_dataframes,ignore_index=True)
london_fhrs_raw = london_fhrs_raw.sort_values(["LocalAuthorityName", "FHRSID"], ignore_index=True)

expected_records = int(london_authorities["EstablishmentCount"].sum())
downloaded_records = len(london_fhrs_raw)

print(f"\nExpected records: {expected_records:,}")
print(f"Downloaded records: {downloaded_records:,}")
print("Expected record count matches downloaded record count:", expected_records == downloaded_records)

print(f"\nTotal London records downloaded: {downloaded_records}")

print(f"Total columns returned: {len(london_fhrs_raw.columns)}")

Barking and Dagenham: 8/8 pages successfully downloaded with 1,448 records found.
Bexley: 9/9 pages successfully downloaded with 1,772 records found.
Barnet: 15/15 pages successfully downloaded with 2,867 records found.
Brent: 13/13 pages successfully downloaded with 2,490 records found.
Bromley: 13/13 pages successfully downloaded with 2,459 records found.
City of London Corporation: 9/9 pages successfully downloaded with 1,800 records found.
Camden: 22/22 pages successfully downloaded with 4,275 records found.
Croydon: 16/16 pages successfully downloaded with 3,144 records found.
Ealing: 19/19 pages successfully downloaded with 3,708 records found.
Greenwich: 12/12 pages successfully downloaded with 2,382 records found.
Enfield: 12/12 pages successfully downloaded with 2,379 records found.
Hammersmith and Fulham: 10/10 pages successfully downloaded with 1,992 records found.
Haringey: 10/10 pages successfully downloaded with 1,958 records found.
Hackney: 14/14 pages successfully downl

## Saving full London raw dataset snapshot

In [12]:
raw_filename = (f"london_fhrs_raw_{SNAPSHOT_DATE}.csv")

raw_path = RAW_FOLDER / raw_filename

london_fhrs_raw.to_csv(raw_path,index=False,)

print(f"Raw snapshot saved to:{raw_path}")

Raw snapshot saved to:..\data\raw\london_fhrs_raw_2026-07-23.csv


## Create working dataset

Reducing number of columns used in the actual working dataset to just the required columns.

In [22]:
required_columns = [
    "FHRSID",
    "BusinessName",
    "BusinessType",
    "PostCode",
    "LocalAuthorityName",
    "geocode.longitude",
    "geocode.latitude",
]

missing_columns = []

for column in required_columns:
    if column not in london_fhrs_raw.columns:
        missing_columns.append(column)

if missing_columns:
    raise RuntimeError(f"Required column missing from API response: {missing_columns}")

print(f"No columns missing. All {len(required_columns)} required columns were found.")

london_fhrs_req_columns = (london_fhrs_raw[required_columns].copy())

print(f"Dataset shape: {london_fhrs_req_columns.shape}")
      
london_fhrs_req_columns.head()

No columns missing. All 7 required columns were found.
Dataset shape: (81085, 7)


,FHRSID,BusinessName,BusinessType,PostCode,LocalAuthorityName,geocode.longitude,geocode.latitude
0,56458,Holibrook House Childrens Home,Caring Premises,IG11 8RB,Barking and Dagenham,0.081816,51.543048
1,66309,BP Service Station,Retailers - other,RM10 7UU,Barking and Dagenham,0.161954,51.550546
2,91426,Playdays Nursery and PreSchool,Caring Premises,RM8 1DD,Barking and Dagenham,0.134142,51.561604
3,91591,Co-op Welcome,Retailers - supermarkets/hypermarkets,RM9 4TP,Barking and Dagenham,0.127901,51.539792
4,95942,K.F.C.,Takeaway/sandwich shop,IG11 8EB,Barking and Dagenham,0.080525,51.539087


## Final checks on dataset

Checking mainly for row count matches, with cleaning done at a later stage.

In [23]:
print(f"Raw and required dataset row counts match:", len(london_fhrs_raw) == len(london_fhrs_req_columns))

print(f"Missing FHRS IDs: {london_fhrs_req_columns["FHRSID"].isna().sum()}")

print(f"Duplicate FHRS IDs: {london_fhrs_req_columns["FHRSID"].duplicated().sum()}")

Raw and required dataset row counts match: True
Missing FHRS IDs: 0
Duplicate FHRS IDs: 5


## Saving working dataset

In [ ]:
raw_snapshot_path = Path(f"../data/business/raw/london_fhrs_raw_{SNAPSHOT_DATE}.csv")

working_dataset_path = Path(f"../data/business/interim/london_fhrs_req_columns_{SNAPSHOT_DATE}.csv")

required_columns = ["FHRSID",
                    "BusinessName",
                    "BusinessType",
                    "PostCode",
                    "LocalAuthorityName",
                    "geocode.longitude",
                    "geocode.latitude"]

london_fhrs_raw = pd.read_csv(raw_snapshot_path, low_memory=False)

london_fhrs_req_columns = london_fhrs_raw[required_columns].copy()

london_fhrs_req_columns.to_csv(working_dataset_path,index=False)

print(f"Raw snapshot rows: {len(london_fhrs_raw)}")
print(f"Working dataset rows: {len(london_fhrs_req_columns)}")
print(f"Saved to: {working_dataset_path}")

Raw snapshot rows: 81085
Working dataset rows: 81085
Saved to: ..\data\business\interim\london_fhrs_req_columns_2026-07-23.csv
